In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.set_printoptions(suppress=True)

In [11]:
from sbi_cc.simulators.water.water_cluster import water_cluster
from sbi_cc.utils.molecule_utils import (
    build_pyscf_molecule,
    compute_dft,
    compute_rhf,
    compute_mp2,
    compute_cc
)

## Meta-variables

In [12]:
basis = 'cc-pVTZ'
num_molecules = 20
grid_level = 7
tolerance = 1e-10

## Water cluster geometries

In [13]:
water = build_pyscf_molecule(
    geometry=water_cluster(num_molecules=num_molecules),
    basis=basis
)

radius: 13.416407864998739


In [14]:
water._atom

[('O', [-18.79096706722664, -9.409961531123713, 0.0]),
 ('H', [-18.690511911409946, -7.635514360125872, 0.0]),
 ('H', [-17.26147721449075, -10.315176859855143, 0.0]),
 ('O', [-3.8983142485338425, 11.019759372278516, 0.0]),
 ('H', [-5.277825903383872, 9.576793499146941, 0.0]),
 ('H', [-4.582271871113488, 12.895233948860302, 0.0]),
 ('O', [-13.371406936168945, -14.329335884018958, 0.0]),
 ('H', [-11.810637549676509, -15.1751745561192, 0.0]),
 ('H', [-14.493436345447357, -15.705015404018663, 0.0]),
 ('O', [22.309288507144707, -2.9259828433283017, 0.0]),
 ('H', [22.05442515086683, -1.134943155700368, 0.0]),
 ('H', [23.77448269056561, -3.9871070103799955, 0.0]),
 ('O', [-2.3451734425107937, -21.151033498401976, 0.0]),
 ('H', [-0.6182226777368957, -21.715993494721506, 0.0]),
 ('H', [-2.9094422443117858, -22.87821022957588, 0.0]),
 ('O', [2.0248560767500594, 23.37482087361732, 0.0]),
 ('H', [1.2068719740427976, 24.966547333173274, 0.0]),
 ('H', [3.727627009154288, 23.925513406789484, 0.0]),
 

## DFT

In [15]:
%%time
dft = compute_dft(mol=water, grid_level=grid_level, tolerance=tolerance, density_fit=True)

SCF not converged.
SCF energy = -1510.96624221709
SCF not converged.
SCF energy = -1520.36139688829
CPU times: user 4h 22min 43s, sys: 15min 24s, total: 4h 38min 8s
Wall time: 13min 30s


In [26]:
for k, v in dft.items():
    print(k, v.shape if isinstance(v, np.ndarray) else v)

dm_dft_ao (1160, 1160)
C_dft (1160, 1160)
dm_dft_mo (1160, 1160)
rho_dft (3342152,)
n_electrons_dft 200.00000188406534
e_dft -1521.7062150363301
e_pbe -1521.7062150363301
converged False
grids <pyscf.dft.gen_grid.Grids object at 0x72cbfb30d310>
ni <pyscf.dft.numint.NumInt object at 0x72cbfb7fdc40>


So far, everything seems fine. But this is assuming that we are using sto-3g. With higher-fidelity basis sets, this becomes difficult for a workstation computer. Expert summary statistics will be needed.

## RHF

In [27]:
%%time
rhf = compute_rhf(mol=water, grids=dft["grids"], tolerance=tolerance, ni=dft["ni"])

converged SCF energy = -1515.30546269828
CPU times: user 26min 14s, sys: 24.9 s, total: 26min 39s
Wall time: 1min 16s


In [28]:
for k, v in rhf.items():
    print(k, v.shape if isinstance(v, np.ndarray) else v)

e_rhf -1515.3054626982755
dm_rhf_ao (1160, 1160)
C_rhf (1160, 1160)
dm_rhf_mo (1160, 1160)
rho_rhf (3342152,)
n_electrons_rhf 200.00000038742905
converged True
rhf <pyscf.df.df_jk.DFRHF object at 0x72cbfb0b7260>


In [29]:
rhf.keys()

dict_keys(['e_rhf', 'dm_rhf_ao', 'C_rhf', 'dm_rhf_mo', 'rho_rhf', 'n_electrons_rhf', 'converged', 'rhf'])

Computation is easier here.

## MP2

In [31]:
%%time
mp2 = compute_mp2(mol=water, rhf=rhf["rhf"], grids=dft["grids"], tolerance=tolerance, ni=dft["ni"])


ERROR: Insufficient memory for holding t2 incore. Please rerun with `with_t2 = False`.

CPU times: user 56.7 s, sys: 1.26 s, total: 58 s
Wall time: 3 s


ERROR: Insufficient memory for holding t2 incore. Please rerun with `with_t2 = False`.


MemoryError: 

In [27]:
for k, v in mp2.items():
    print(k, v.shape if isinstance(v, np.ndarray) else v)

e_tot_mp2 -749.4793192452928
e_corr_mp2 -0.35012597386680644
dm_mp2_ao (70, 70)
C_mp2 (70, 70)
dm_mp2_mo (70, 70)
rho_mp2 (900584,)
n_electrons_mp2 100.00000022096467
t2_a (50, 100)
t2_b (20, 100)
t2_c (1000, 100)
t2_fact_rec (50, 20, 1000)
max_t2_t2_rec 1.0061396160665481e-16
max_t2_fact_rec 0.9273136235323369
t2_mat (1000, 1000)
eigvals (1000,)
eigvecs (1000, 1000)
R 1000
eigvals_R (1000,)
eigvecs_R (1000, 1000)
t2_fact (50, 20, 1000)
t2_rec (50, 50, 20, 20)


Again, the factorized matrices are looking BIG.

## CC

Last but not least.

In [29]:
cc = compute_cc(mol=water, rhf=rhf["rhf"], grids=grids, ni=ni)

E(CCSD) = -749.6113686149329  E_corr = -0.4821753435069211
CCSD(T) correction = -0.00165519936188883
max |T - T^T| = 0.0


In [31]:
for k, v in cc.items():
print(k, v.shape if isinstance(v, np.ndarray) else v)

e_tot_cc -749.6113686149329
e_corr_cc -0.4821753435069211
dm_cc_ao (70, 70)
C_cc (70, 70)
dm_cc_mo (70, 70)
rho_cc (900584,)
n_electrons_cc 100.00000024095796
t1 (50, 20)
t2 (50, 50, 20, 20)
t2_mat (1000, 1000)
eigvals (1000,)
eigvecs (1000, 1000)
R 1000
eigvals_R (1000,)
eigvecs_R (1000, 1000)
t2_fact (50, 20, 1000)
t2_rec (50, 50, 20, 20)


In [32]:
from sbi_cc.utils.tensor_utils import update_C

t2_A = mp2['t2_a']
t2_B = mp2['t2_b']
t2_C = update_C(cc['t2_fact'], t2_A, t2_B)

In [33]:
t2_fact_rec = np.einsum('ir,ar,kr->iak', t2_A, t2_B, t2_C)
print("max |T2_factor - T2_rec_factor_mat| =", np.max(np.abs(cc['t2_fact'] - t2_fact_rec)))

max |T2_factor - T2_rec_factor_mat| = 0.925647170309481


In [35]:
print("Factor matrix differences (CCSD - MP2):")
print(f"Occ Factor: {np.linalg.norm(t2_A - mp2["t2_a"])}")
print(f"Vir Factor: {np.linalg.norm(t2_B - mp2["t2_b"])}")
print(f"Cor Factor: {np.linalg.norm(t2_C - mp2["t2_c"])}")

Factor matrix differences (CCSD - MP2):
Occ Factor: 0.0
Vir Factor: 0.0
Cor Factor: 1811.3078697952863
